In [ ]:
# ============================================================
# BLUESTOCK MF - DAY 3 PERFORMANCE ANALYTICS
# ============================================================
#
# Outputs:
#   performance/
#       fund_scorecard.csv
#       alpha_beta.csv
#       daily_returns.csv
#       cagr_comparison.csv
#       drawdown_analysis.csv
#       tracking_error.csv
#       benchmark_comparison_top5.png
#
#   notebooks/
#       Performance_Analytics.ipynb
#
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import linregress

warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================
def find_project_root():

    candidates = []

    if "__file__" in globals():
        candidates.append(
            Path(__file__).resolve().parent
        )

    candidates.append(Path.cwd())

    for candidate in candidates:

        candidate = candidate.resolve()

        if (candidate / "data" / "raw").exists():
            return candidate

    return candidates[0].resolve()


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "performance"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. CONSTANTS
# ============================================================

RISK_FREE_RATE = 0.065
TRADING_DAYS = 252

NAV_FILE = PROCESSED_DIR / "02_nav_history_clean.csv"
BENCHMARK_FILE = RAW_DIR / "10_benchmark_indices.csv"

# Possible benchmark names used in the source data
NIFTY100_NAMES = [
    "Nifty 100",
    "NIFTY 100",
    "NIFTY100",
    "Nifty100",
]

NIFTY50_NAMES = [
    "Nifty 50",
    "NIFTY 50",
    "NIFTY50",
    "Nifty50",
]


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def find_column(df, candidates, required=True):
    """
    Find a dataframe column using case-insensitive matching.
    """

    normalized = {
        str(col).strip().lower().replace(" ", "_"): col
        for col in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "_")

        if key in normalized:
            return normalized[key]

    if required:
        raise KeyError(
            f"Could not find any of {candidates}. "
            f"Available columns: {list(df.columns)}"
        )

    return None


def standardize_columns(df):
    """
    Standardize column names.
    """

    df = df.copy()

    df.columns = [
    str(c)
    .strip()
    .lower()
    .replace(" ", "_")
    .replace("-", "_")
    .replace("/", "_")
    for c in df.columns
]

    return df


def parse_date_column(df):
    """
    Detect and parse date column.
    """

    date_col = find_column(
        df,
        [
            "date",
            "nav_date",
            "business_date",
            "trading_date",
            "as_on_date",
        ],
    )

    df[date_col] = pd.to_datetime(
        df[date_col],
        errors="coerce",
    )

    df = df.dropna(subset=[date_col])

    return df, date_col


def annualized_return_from_daily(daily_returns):
    """
    Annualized return from daily returns.
    """

    daily_returns = daily_returns.dropna()

    if len(daily_returns) == 0:
        return np.nan

    cumulative = (1 + daily_returns).prod()

    years = len(daily_returns) / TRADING_DAYS

    if years <= 0 or cumulative <= 0:
        return np.nan

    return cumulative ** (1 / years) - 1


def calculate_cagr(nav_series, years):
    """
    Calculate CAGR using NAV values approximately n years apart.
    """

    nav_series = nav_series.dropna().sort_index()

    if len(nav_series) < 2:
        return np.nan

    end_date = nav_series.index.max()

    target_date = end_date - pd.DateOffset(years=years)

    available_dates = nav_series.index[
        nav_series.index <= target_date
    ]

    if len(available_dates) == 0:
        return np.nan

    start_date = available_dates[-1]

    start_nav = nav_series.loc[start_date]
    end_nav = nav_series.loc[end_date]

    if start_nav <= 0 or end_nav <= 0:
        return np.nan

    return (end_nav / start_nav) ** (1 / years) - 1


def maximum_drawdown(nav_series):
    """
    Maximum drawdown and associated peak/trough/recovery dates.
    """

    nav_series = nav_series.dropna().sort_index()

    if len(nav_series) == 0:
        return {
            "max_drawdown": np.nan,
            "drawdown_date": pd.NaT,
            "peak_date": pd.NaT,
            "recovery_date": pd.NaT,
        }

    running_max = nav_series.cummax()

    drawdown = nav_series / running_max - 1

    trough_date = drawdown.idxmin()
    max_dd = drawdown.loc[trough_date]

    peak_before_trough = nav_series.loc[
        :trough_date
    ]

    peak_date = peak_before_trough.idxmax()

    peak_value = peak_before_trough.max()

    after_trough = nav_series.loc[trough_date:]

    recovery_dates = after_trough[
        after_trough >= peak_value
    ]

    if len(recovery_dates) > 0:
        recovery_date = recovery_dates.index[0]
    else:
        recovery_date = pd.NaT

    return {
        "max_drawdown": max_dd,
        "drawdown_date": trough_date,
        "peak_date": peak_date,
        "recovery_date": recovery_date,
    }


# ============================================================
# 4. LOAD NAV DATA
# ============================================================

print("=" * 70)
print("BLUESTOCK MF - DAY 3 PERFORMANCE ANALYTICS")
print("=" * 70)

print("\n[1/10] Loading NAV data...")

if not NAV_FILE.exists():
    raise FileNotFoundError(
        f"NAV file not found:\n{NAV_FILE}"
    )

nav = pd.read_csv(NAV_FILE)

nav = standardize_columns(nav)

nav, nav_date_col = parse_date_column(nav)

amfi_col = find_column(
    nav,
    [
        "amfi_code",
        "scheme_code",
        "code",
    ],
)

nav_value_col = find_column(
    nav,
    [
        "nav",
        "nav_value",
        "net_asset_value",
    ],
)

nav[nav_value_col] = pd.to_numeric(
    nav[nav_value_col],
    errors="coerce",
)

nav = nav.dropna(
    subset=[
        amfi_col,
        nav_date_col,
        nav_value_col,
    ]
)

nav = nav[nav[nav_value_col] > 0]

nav = nav.sort_values(
    [amfi_col, nav_date_col]
)

# Remove duplicate scheme/date observations
nav = nav.drop_duplicates(
    subset=[amfi_col, nav_date_col],
    keep="last",
)

print(f"Rows: {len(nav):,}")
print(f"Funds: {nav[amfi_col].nunique():,}")
print(
    f"Date range: "
    f"{nav[nav_date_col].min().date()} "
    f"to "
    f"{nav[nav_date_col].max().date()}"
)


# ============================================================
# 5. CREATE DAILY RETURNS
# ============================================================

print("\n[2/10] Calculating daily returns...")

nav["daily_return"] = (
    nav.groupby(amfi_col)[nav_value_col]
    .pct_change()
)

nav["daily_return"] = nav["daily_return"].replace(
    [np.inf, -np.inf],
    np.nan,
)

daily_returns = nav[
    [
        amfi_col,
        nav_date_col,
        nav_value_col,
        "daily_return",
    ]
].copy()

daily_returns.to_csv(
    OUTPUT_DIR / "daily_returns.csv",
    index=False,
)

print(
    f"Daily return rows: "
    f"{len(daily_returns):,}"
)

print(
    "Return summary:"
)

print(
    daily_returns["daily_return"].describe()
)


# ============================================================
# 6. LOAD BENCHMARK DATA
# ============================================================

print("\n[3/10] Loading benchmark data...")

if not BENCHMARK_FILE.exists():
    raise FileNotFoundError(
        f"Benchmark file not found:\n{BENCHMARK_FILE}"
    )

benchmark = pd.read_csv(BENCHMARK_FILE)

benchmark = standardize_columns(benchmark)

benchmark, benchmark_date_col = parse_date_column(
    benchmark
)

print(
    "Benchmark columns:",
    list(benchmark.columns)
)


# ------------------------------------------------------------
# Identify benchmark name/index column
# ------------------------------------------------------------

benchmark_name_col = find_column(
    benchmark,
    [
        "index_name",
        "benchmark",
        "index",
        "name",
        "scheme_name",
        "symbol",
    ],
    required=False,
)


# ------------------------------------------------------------
# Identify benchmark value column
# ------------------------------------------------------------

benchmark_value_col = find_column(
    benchmark,
    [
        "close",
        "closing_price",
        "close_value",
        "index_value",
        "value",
        "nav",
        "price",
    ],
    required=False,
)


# ------------------------------------------------------------
# Case where benchmark CSV is wide
# ------------------------------------------------------------

if (
    benchmark_name_col is None
    and benchmark_value_col is None
):

    possible_nifty100 = [
        c for c in benchmark.columns
        if str(c).strip().lower().replace(" ", "")
        in {
            "nifty100",
            "nifty_100",
        }
    ]

    possible_nifty50 = [
        c for c in benchmark.columns
        if str(c).strip().lower().replace(" ", "")
        in {
            "nifty50",
            "nifty_50",
        }
    ]

    if possible_nifty100:
        nifty100_col = possible_nifty100[0]
    else:
        nifty100_col = None

    if possible_nifty50:
        nifty50_col = possible_nifty50[0]
    else:
        nifty50_col = None

else:

    # --------------------------------------------------------
    # Long format
    # --------------------------------------------------------

    benchmark[benchmark_value_col] = pd.to_numeric(
        benchmark[benchmark_value_col],
        errors="coerce",
    )

    benchmark_name_values = (
        benchmark[benchmark_name_col]
        .astype(str)
        .str.strip()
    )

    # Find actual Nifty names
    nifty100_mask = benchmark_name_values.str.lower().isin(
        [x.lower() for x in NIFTY100_NAMES]
    )

    nifty50_mask = benchmark_name_values.str.lower().isin(
        [x.lower() for x in NIFTY50_NAMES]
    )

    if nifty100_mask.any():
        nifty100 = benchmark.loc[
            nifty100_mask,
            [
                benchmark_date_col,
                benchmark_value_col,
            ],
        ].copy()

        nifty100 = nifty100.rename(
            columns={
                benchmark_date_col: "date",
                benchmark_value_col: "nifty100",
            }
        )

    else:
        nifty100 = pd.DataFrame(
            columns=["date", "nifty100"]
        )

    if nifty50_mask.any():
        nifty50 = benchmark.loc[
            nifty50_mask,
            [
                benchmark_date_col,
                benchmark_value_col,
            ],
        ].copy()

        nifty50 = nifty50.rename(
            columns={
                benchmark_date_col: "date",
                benchmark_value_col: "nifty50",
            }
        )

    else:
        nifty50 = pd.DataFrame(
            columns=["date", "nifty50"]
        )

    nifty100["date"] = pd.to_datetime(
        nifty100["date"],
        errors="coerce",
    )

    nifty50["date"] = pd.to_datetime(
        nifty50["date"],
        errors="coerce",
    )

    nifty100["nifty100"] = pd.to_numeric(
        nifty100["nifty100"],
        errors="coerce",
    )

    nifty50["nifty50"] = pd.to_numeric(
        nifty50["nifty50"],
        errors="coerce",
    )

    nifty100 = nifty100.dropna(
        subset=["date", "nifty100"]
    )

    nifty50 = nifty50.dropna(
        subset=["date", "nifty50"]
    )

    nifty100 = nifty100.drop_duplicates(
        "date"
    )

    nifty50 = nifty50.drop_duplicates(
        "date"
    )

    nifty100 = nifty100.sort_values("date")
    nifty50 = nifty50.sort_values("date")

    nifty100["nifty100_return"] = (
        nifty100["nifty100"].pct_change()
    )

    nifty50["nifty50_return"] = (
        nifty50["nifty50"].pct_change()
    )

    print(
        f"Nifty 100 observations: {len(nifty100):,}"
    )

    print(
        f"Nifty 50 observations: {len(nifty50):,}"
    )


# ------------------------------------------------------------
# Handle wide benchmark format
# ------------------------------------------------------------

if (
    benchmark_name_col is None
    and benchmark_value_col is None
):

    if nifty100_col is not None:

        nifty100 = benchmark[
            [
                benchmark_date_col,
                nifty100_col,
            ]
        ].copy()

        nifty100.columns = [
            "date",
            "nifty100",
        ]

        nifty100["nifty100"] = pd.to_numeric(
            nifty100["nifty100"],
            errors="coerce",
        )

        nifty100 = nifty100.dropna(
            subset=["date", "nifty100"]
        )

        nifty100 = nifty100.sort_values(
            "date"
        )

        nifty100["nifty100_return"] = (
            nifty100["nifty100"].pct_change()
        )

    else:

        nifty100 = pd.DataFrame(
            columns=[
                "date",
                "nifty100",
                "nifty100_return",
            ]
        )

    if nifty50_col is not None:

        nifty50 = benchmark[
            [
                benchmark_date_col,
                nifty50_col,
            ]
        ].copy()

        nifty50.columns = [
            "date",
            "nifty50",
        ]

        nifty50["nifty50"] = pd.to_numeric(
            nifty50["nifty50"],
            errors="coerce",
        )

        nifty50 = nifty50.dropna(
            subset=["date", "nifty50"]
        )

        nifty50 = nifty50.sort_values(
            "date"
        )

        nifty50["nifty50_return"] = (
            nifty50["nifty50"].pct_change()
        )

    else:

        nifty50 = pd.DataFrame(
            columns=[
                "date",
                "nifty50",
                "nifty50_return",
            ]
        )


# ============================================================
# 7. FUND-LEVEL PERFORMANCE METRICS
# ============================================================

print("\n[4/10] Calculating fund performance metrics...")

fund_records = []
alpha_beta_records = []
drawdown_records = []

for amfi_code, group in nav.groupby(amfi_col):

    group = group.sort_values(
        nav_date_col
    ).copy()

    series = group.set_index(
        nav_date_col
    )[nav_value_col]

    returns = group["daily_return"].dropna()

    if len(returns) == 0:
        continue

    # --------------------------------------------------------
    # CAGR
    # --------------------------------------------------------

    cagr_1y = calculate_cagr(
        series,
        1,
    )

    cagr_3y = calculate_cagr(
        series,
        3,
    )

    cagr_5y = calculate_cagr(
        series,
        5,
    )

    # --------------------------------------------------------
    # Sharpe
    # --------------------------------------------------------

    daily_rf = (
        (1 + RISK_FREE_RATE)
        ** (1 / TRADING_DAYS)
        - 1
    )

    excess_returns = returns - daily_rf

    std_return = returns.std()

    if std_return > 0:

        sharpe = (
            excess_returns.mean()
            / std_return
            * np.sqrt(TRADING_DAYS)
        )

    else:
        sharpe = np.nan

    # --------------------------------------------------------
    # Sortino
    # --------------------------------------------------------

    downside_returns = returns[
        returns < 0
    ]

    downside_std = downside_returns.std()

    if (
        downside_std is not None
        and pd.notna(downside_std)
        and downside_std > 0
    ):

        sortino = (
            excess_returns.mean()
            / downside_std
            * np.sqrt(TRADING_DAYS)
        )

    else:
        sortino = np.nan

    # --------------------------------------------------------
    # Maximum Drawdown
    # --------------------------------------------------------

    dd = maximum_drawdown(series)

    fund_records.append(
        {
            "amfi_code": amfi_code,
            "start_date": series.index.min(),
            "end_date": series.index.max(),
            "observations": len(series),
            "cagr_1y": cagr_1y,
            "cagr_3y": cagr_3y,
            "cagr_5y": cagr_5y,
            "sharpe_ratio": sharpe,
            "sortino_ratio": sortino,
            "max_drawdown": dd["max_drawdown"],
            "drawdown_date": dd["drawdown_date"],
            "drawdown_peak_date": dd["peak_date"],
            "drawdown_recovery_date": dd[
                "recovery_date"
            ],
        }
    )

fund_metrics = pd.DataFrame(
    fund_records
)

print(
    f"Funds analyzed: "
    f"{len(fund_metrics):,}"
)


# ============================================================
# 8. ALPHA / BETA AGAINST NIFTY 100
# ============================================================

print("\n[5/10] Calculating Alpha and Beta...")

for amfi_code, group in nav.groupby(amfi_col):

    fund_returns = group[
        [
            nav_date_col,
            "daily_return",
        ]
    ].copy()

    fund_returns = fund_returns.rename(
        columns={
            nav_date_col: "date",
            "daily_return": "fund_return",
        }
    )

    merged = fund_returns.merge(
        nifty100[
            [
                "date",
                "nifty100_return",
            ]
        ],
        on="date",
        how="inner",
    )

    merged = merged.dropna(
        subset=[
            "fund_return",
            "nifty100_return",
        ]
    )

    if len(merged) < 2:

        alpha = np.nan
        beta = np.nan
        r_squared = np.nan
        p_value = np.nan
        std_err = np.nan

    else:

        result = linregress(
            merged["nifty100_return"],
            merged["fund_return"],
        )

        beta = result.slope

        alpha_daily = result.intercept

        alpha = (
            alpha_daily
            * TRADING_DAYS
        )

        r_squared = (
            result.rvalue ** 2
        )

        p_value = result.pvalue
        std_err = result.stderr

    # --------------------------------------------------------
    # Tracking error
    # --------------------------------------------------------

    merged["active_return"] = (
        merged["fund_return"]
        - merged["nifty100_return"]
    )

    tracking_error_nifty100 = (
        merged["active_return"].std()
        * np.sqrt(TRADING_DAYS)
    )

    alpha_beta_records.append(
        {
            "amfi_code": amfi_code,
            "alpha": alpha,
            "beta": beta,
            "r_squared": r_squared,
            "p_value": p_value,
            "std_err": std_err,
            "tracking_error_nifty100":
                tracking_error_nifty100,
            "nifty100_observations":
                len(merged),
        }
    )


alpha_beta = pd.DataFrame(
    alpha_beta_records
)


# ============================================================
# 9. MERGE PERFORMANCE TABLES
# ============================================================

print("\n[6/10] Building comparison table...")

performance = fund_metrics.merge(
    alpha_beta,
    on="amfi_code",
    how="left",
)

# ------------------------------------------------------------
# Expense ratio from scheme performance
# ------------------------------------------------------------

scheme_file = (
    PROCESSED_DIR
    / "07_scheme_performance_clean.csv"
)

if scheme_file.exists():

    scheme = pd.read_csv(
        scheme_file
    )

    scheme = standardize_columns(
        scheme
    )

    scheme_amfi = find_column(
        scheme,
        [
            "amfi_code",
            "scheme_code",
            "code",
        ],
    )

    expense_col = find_column(
        scheme,
        [
            "expense_ratio",
            "expense",
            "expense_ratio_pct",
        ],
        required=False,
    )

    if expense_col is not None:

        expense = scheme[
            [
                scheme_amfi,
                expense_col,
            ]
        ].copy()

        expense.columns = [
            "amfi_code",
            "expense_ratio",
        ]

        expense["expense_ratio"] = pd.to_numeric(
            expense["expense_ratio"],
            errors="coerce",
        )

        expense = expense.drop_duplicates(
            "amfi_code"
        )

        performance = performance.merge(
            expense,
            on="amfi_code",
            how="left",
        )

    else:

        performance["expense_ratio"] = np.nan

else:

    performance["expense_ratio"] = np.nan


# ============================================================
# 10. FUND SCORECARD
# ============================================================

print("\n[7/10] Creating 0-100 fund scorecard...")

score = performance.copy()


# ------------------------------------------------------------
# Rank helper
# ------------------------------------------------------------

def percentile_score(series, higher_is_better=True):

    valid = series.notna()

    result = pd.Series(
        np.nan,
        index=series.index,
        dtype=float,
    )

    if valid.sum() == 0:
        return result

    if higher_is_better:

        result.loc[valid] = (
            series[valid].rank(
                method="average",
                pct=True,
            )
            * 100
        )

    else:

        result.loc[valid] = (
            series[valid]
            .rank(
                method="average",
                pct=True,
                ascending=False,
            )
            * 100
        )

    return result


# 3Y return: higher is better
score["return_3y_score"] = percentile_score(
    score["cagr_3y"],
    higher_is_better=True,
)

# Sharpe: higher is better
score["sharpe_score"] = percentile_score(
    score["sharpe_ratio"],
    higher_is_better=True,
)

# Alpha: higher is better
score["alpha_score"] = percentile_score(
    score["alpha"],
    higher_is_better=True,
)

# Expense ratio: lower is better
score["expense_score"] = percentile_score(
    score["expense_ratio"],
    higher_is_better=False,
)

# Max drawdown:
# less negative is better
score["drawdown_score"] = percentile_score(
    score["max_drawdown"],
    higher_is_better=True,
)


# ------------------------------------------------------------
# Composite score
# ------------------------------------------------------------

score["fund_score"] = (
    0.30 * score["return_3y_score"]
    + 0.25 * score["sharpe_score"]
    + 0.20 * score["alpha_score"]
    + 0.15 * score["expense_score"]
    + 0.10 * score["drawdown_score"]
)

score["fund_score"] = score[
    "fund_score"
].clip(
    lower=0,
    upper=100,
)

score = score.sort_values(
    "fund_score",
    ascending=False,
)


# ============================================================
# 11. SAVE SCORECARD
# ============================================================

scorecard_columns = [
    "amfi_code",
    "cagr_1y",
    "cagr_3y",
    "cagr_5y",
    "sharpe_ratio",
    "sortino_ratio",
    "alpha",
    "beta",
    "expense_ratio",
    "max_drawdown",
    "drawdown_date",
    "drawdown_peak_date",
    "drawdown_recovery_date",
    "tracking_error_nifty100",
    "return_3y_score",
    "sharpe_score",
    "alpha_score",
    "expense_score",
    "drawdown_score",
    "fund_score",
]

scorecard = score[
    [
        c
        for c in scorecard_columns
        if c in score.columns
    ]
].copy()

scorecard.to_csv(
    OUTPUT_DIR / "fund_scorecard.csv",
    index=False,
)

performance[
    [
        c
        for c in [
            "amfi_code",
            "alpha",
            "beta",
            "r_squared",
            "p_value",
            "std_err",
            "tracking_error_nifty100",
            "nifty100_observations",
        ]
        if c in performance.columns
    ]
].to_csv(
    OUTPUT_DIR / "alpha_beta.csv",
    index=False,
)

performance.to_csv(
    OUTPUT_DIR / "cagr_comparison.csv",
    index=False,
)

fund_metrics.to_csv(
    OUTPUT_DIR / "drawdown_analysis.csv",
    index=False,
)


# ============================================================
# 12. BENCHMARK COMPARISON
# ============================================================

print("\n[8/10] Creating benchmark comparison...")

# Top 5 based on fund score
top5 = score.head(5)[
    "amfi_code"
].tolist()

# ------------------------------------------------------------
# Last 3 years
# ------------------------------------------------------------

overall_end = nav[nav_date_col].max()

three_year_start = (
    overall_end
    - pd.DateOffset(years=3)
)

nav_3y = nav[
    nav[nav_date_col] >= three_year_start
].copy()


# ------------------------------------------------------------
# Build indexed NAV series
# ------------------------------------------------------------

plot_data = pd.DataFrame()

for code in top5:

    temp = nav_3y[
        nav_3y[amfi_col] == code
    ][
        [
            nav_date_col,
            nav_value_col,
        ]
    ].copy()

    if len(temp) == 0:
        continue

    temp = temp.sort_values(
        nav_date_col
    )

    temp = temp.set_index(
        nav_date_col
    )

    indexed = (
        temp[nav_value_col]
        / temp[nav_value_col].iloc[0]
        * 100
    )

    plot_data[
        str(code)
    ] = indexed


# ------------------------------------------------------------
# Add Nifty 50 and Nifty 100
# ------------------------------------------------------------

if len(nifty100) > 0:

    temp = nifty100[
        nifty100["date"] >= three_year_start
    ].copy()

    temp = temp.sort_values("date")

    if len(temp) > 0:

        temp = temp.set_index("date")

        plot_data[
            "Nifty 100"
        ] = (
            temp["nifty100"]
            / temp["nifty100"].iloc[0]
            * 100
        )


if len(nifty50) > 0:

    temp = nifty50[
        nifty50["date"] >= three_year_start
    ].copy()

    temp = temp.sort_values("date")

    if len(temp) > 0:

        temp = temp.set_index("date")

        plot_data[
            "Nifty 50"
        ] = (
            temp["nifty50"]
            / temp["nifty50"].iloc[0]
            * 100
        )


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(
    figsize=(15, 8)
)

for column in plot_data.columns:

    if column in [
        "Nifty 50",
        "Nifty 100",
    ]:

        plt.plot(
            plot_data.index,
            plot_data[column],
            linewidth=2.5,
            linestyle="--",
            label=column,
        )

    else:

        plt.plot(
            plot_data.index,
            plot_data[column],
            linewidth=1.5,
            label=f"Fund {column}",
        )


plt.title(
    "Top 5 Mutual Funds vs Nifty 50 and Nifty 100 - 3 Years"
)

plt.xlabel("Date")
plt.ylabel("Indexed Value (Start = 100)")

plt.grid(
    alpha=0.3
)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.tight_layout()

chart_path = (
    OUTPUT_DIR
    / "benchmark_comparison_top5.png"
)

plt.savefig(
    chart_path,
    dpi=200,
    bbox_inches="tight",
)

plt.close()


# ============================================================
# 13. TRACKING ERROR FOR NIFTY 50
# ============================================================

print("\n[9/10] Calculating Nifty 50 tracking error...")

tracking_records = []

for code in top5:

    fund_returns = nav_3y[
        nav_3y[amfi_col] == code
    ][
        [
            nav_date_col,
            "daily_return",
        ]
    ].copy()

    fund_returns = fund_returns.rename(
        columns={
            nav_date_col: "date",
            "daily_return": "fund_return",
        }
    )

    merged_50 = fund_returns.merge(
        nifty50[
            [
                "date",
                "nifty50_return",
            ]
        ],
        on="date",
        how="inner",
    )

    merged_50 = merged_50.dropna()

    if len(merged_50) > 1:

        tracking_error_50 = (
            (
                merged_50["fund_return"]
                - merged_50["nifty50_return"]
            ).std()
            * np.sqrt(TRADING_DAYS)
        )

    else:

        tracking_error_50 = np.nan

    tracking_records.append(
        {
            "amfi_code": code,
            "tracking_error_nifty50":
                tracking_error_50,
            "observations":
                len(merged_50),
        }
    )


tracking_error = pd.DataFrame(
    tracking_records
)

tracking_error.to_csv(
    OUTPUT_DIR / "tracking_error.csv",
    index=False,
)


# ============================================================
# 14. CREATE NOTEBOOK
# ============================================================

print("\n[10/10] Creating Performance_Analytics.ipynb...")

notebook_path = (
    NOTEBOOK_DIR
    / "Performance_Analytics.ipynb"
)

markdown_cell = {
    "cell_type": "markdown",
    "metadata": {},
    "source": [
        "# Bluestock MF - Performance Analytics\n",
        "\n",
        "Day 3 performance analytics covering daily returns, CAGR, Sharpe, Sortino, Alpha, Beta, maximum drawdown, fund scorecard and benchmark comparison.\n",
        "\n",
        "Risk-free rate proxy: 6.5%.\n",
        "\n",
        "Benchmark: Nifty 100 for Alpha/Beta and tracking error; Nifty 50 and Nifty 100 for the 3-year comparison chart."
    ],
}

code_cell = {
    "cell_type": "code",
    "execution_count": None,
    "metadata": {},
    "outputs": [],
    "source": [
        "from pathlib import Path\n",
        "import pandas as pd\n",
        "import matplotlib.pyplot as plt\n",
        "\n",
        "PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()\n",
        "OUTPUT_DIR = PROJECT_ROOT / 'performance'\n",
        "\n",
        "scorecard = pd.read_csv(OUTPUT_DIR / 'fund_scorecard.csv')\n",
        "alpha_beta = pd.read_csv(OUTPUT_DIR / 'alpha_beta.csv')\n",
        "cagr = pd.read_csv(OUTPUT_DIR / 'cagr_comparison.csv')\n",
        "\n",
        "display(scorecard.head(10))\n",
        "display(alpha_beta.head(10))\n        ",
    ],
}

# Fix notebook source formatting
code_cell["source"] = [
    line if line.endswith("\n") else line + "\n"
    for line in code_cell["source"]
]

notebook = {
    "cells": [
        markdown_cell,
        code_cell,
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3",
        },
        "language_info": {
            "name": "python",
            "version": "3.x",
        },
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}

with open(
    notebook_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        notebook,
        f,
        indent=2,
    )


# ============================================================
# 15. VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("DAY 3 VALIDATION")
print("=" * 70)

print(
    f"[OK] Funds analyzed: "
    f"{len(performance):,}"
)

print(
    f"[OK] Daily return file: "
    f"{OUTPUT_DIR / 'daily_returns.csv'}"
)

print(
    f"[OK] CAGR comparison: "
    f"{OUTPUT_DIR / 'cagr_comparison.csv'}"
)

print(
    f"[OK] Alpha/Beta: "
    f"{OUTPUT_DIR / 'alpha_beta.csv'}"
)

print(
    f"[OK] Fund scorecard: "
    f"{OUTPUT_DIR / 'fund_scorecard.csv'}"
)

print(
    f"[OK] Drawdown analysis: "
    f"{OUTPUT_DIR / 'drawdown_analysis.csv'}"
)

print(
    f"[OK] Tracking error: "
    f"{OUTPUT_DIR / 'tracking_error.csv'}"
)

print(
    f"[OK] Benchmark chart: "
    f"{chart_path}"
)

print(
    f"[OK] Notebook: "
    f"{notebook_path}"
)

print("\nTop scorecard rows:")
print(
    scorecard.head(10).to_string(
        index=False
    )
)

print("\n" + "=" * 70)
print("DAY 3 COMPLETE")
print("=" * 70)


BLUESTOCK MF - DAY 3 PERFORMANCE ANALYTICS

[1/10] Loading NAV data...
Rows: 46,000
Funds: 40
Date range: 2022-01-02 to 2026-12-05

[2/10] Calculating daily returns...
Daily return rows: 46,000
Return summary:
count    45960.000000
mean         0.001912
std          0.050842
min         -0.477980
25%         -0.008340
50%          0.000947
75%          0.012524
max          0.518264
Name: daily_return, dtype: float64

[3/10] Loading benchmark data...
Benchmark columns: ['date', 'index_name', 'close_value']
Nifty 100 observations: 1,150
Nifty 50 observations: 1,150

[4/10] Calculating fund performance metrics...
Funds analyzed: 40

[5/10] Calculating Alpha and Beta...

[6/10] Building comparison table...

[7/10] Creating 0-100 fund scorecard...

[8/10] Creating benchmark comparison...

[9/10] Calculating Nifty 50 tracking error...

[10/10] Creating Performance_Analytics.ipynb...


DAY 3 VALIDATION
[OK] Funds analyzed: 40
[OK] Daily return file: C:\Users\javee\Bluestock internship\Capsto